## PDF RAG

Steps:
- Ingesting PDF Files
- Extracting Text form PDF and split into small chunks
- Embed the chunks
- Save it to VDB
- Perform sim search on VDB
- Reterive the docs, present to the use

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

In [3]:
import sys
import logging
logging.basicConfig(level=logging.INFO)

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [5]:
FILE_PATH = r'../data/llm_guide.pdf'
MODEL_NAME = 'gpt-5.1'
EMBEDDING_MODEL = 'text-embedding-3-small'
VECTOR_STORE = 'simple-rag'
PERSIST_DIRECTORY = "./simple-rag-db"

In [6]:
llm = ChatOpenAI(
    api_key = API_KEY,
    base_url = BASE_URL,
    model = MODEL_NAME
)
logging.info('LLM initiallized')
embedding_model = OpenAIEmbeddings(
    api_key= API_KEY,
    base_url = BASE_URL,
    model = EMBEDDING_MODEL
)
logging.info('Embeddings initiallized')

INFO:root:LLM initiallized
INFO:root:Embeddings initiallized


In [7]:
if os.path.exists(FILE_PATH):
    loader = PyPDFLoader(FILE_PATH)
    documents = loader.load()
    logging.info('PDF loaded successfully')
else:
    logging.error(f"PDF file not found at path: {FILE_PATH}")
    sys.exit()

INFO:root:PDF loaded successfully


In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap = 100
)

chunks = text_splitter.split_documents(documents)
logging.info('Documnts splitting chunks')

INFO:root:Documnts splitting chunks


In [21]:
print(len(chunks))

36


In [22]:
if os.path.exists(PERSIST_DIRECTORY):
    vector_store = Chroma(
        collection_name = VECTOR_STORE,
        embedding_function = embedding_model,
        persist_directory = PERSIST_DIRECTORY
    )

    existing_docs = vector_store.get()
    existing_count = len(existing_docs['ids'])
    new_count = len(chunks)

    if existing_count != new_count:
        logging.info('Updating vector sotore')
        vector_store = Chroma(
            collection_name = VECTOR_STORE,
            embedding_function = embedding_model,
            persist_directory = PERSIST_DIRECTORY
        )
        vector_store.add_documents(chunks)
        logging.info('Vector store updated')
    
else:
    logging.info('Vector store is up to date')
    vector_store = Chroma(
        collection_name = VECTOR_STORE,
        embedding_function = embedding_model,
        persist_directory = PERSIST_DIRECTORY
    )
    vector_store.add_documents(chunks)
    logging.info('Vector store createed successfully')

INFO:root:Updating vector sotore


INFO:httpx:HTTP Request: POST https://lkm-ai-az-openai.openai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"
INFO:root:Vector store updated


In [23]:
retriever = vector_store.as_retriever(
    sarch_type = 'similarity',
    search_kwargs = {'k':3}
)
logging.info('Query Reteriver created')

INFO:root:Query Reteriver created


In [24]:
query="What role did the release of ChatGPT play in the public awareness and accessibility of LLMs?"

relevant_docs = retriever.invoke(query)
logging.info("retriever invoked.")
    

INFO:httpx:HTTP Request: POST https://lkm-ai-az-openai.openai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"
INFO:root:retriever invoked.


In [25]:
for i, doc in enumerate(relevant_docs, 1):
    print(f'Documetns {i}: {doc.page_content}')

Documetns 1: Proprietary services
As the first widely available LLM powered service, OpenAI’s ChatGPT was the 
explosive charge that brought LLMs into the mainstream. ChatGPT provides 
a nice user interface (or API) where users can feed prompts to one of many 
models (GPT-3.5, GPT-4, and more) and typically get a fast response. These are 
among the highest-performing models, trained on enormous data sets, and are 
capable of extremely complex tasks both from a technical standpoint, such as
Documetns 2: language-related tasks.
 
2022  
ChatGPT is launched, which turns GPT-3 and similar models into  
a service that is widely accessible to users through a web interface  
and kicks off a huge increase in public awareness of LLMs and  
generative AI.
 
2023  
Open source LLMs begin showing increasingly impressive results  
with releases such as Dolly 2.0, LLaMA, Alpaca and Vicuna.  
GPT-4 is also released, setting a new benchmark for both parameter  
size and performance.
Documetns 3: techn

In [27]:
combined_input = (
    'Here are some docs that might help answer the question: ' + 
    query + 
    '\n\nRelevant Documents:\n' + 
    '\n\n'.join([doc.page_content for doc in relevant_docs]) + 
    '\n\nPlease provide an answer based only on the provided docs. If the answer is not found in the documents, respond with "I\'m not sure".'
)

In [30]:
messsages = [
    SystemMessage("You are a helpful assistant!"),
    HumanMessage(content = combined_input)
]

result= llm.invoke(messsages)
print(result.content)

INFO:httpx:HTTP Request: POST https://lkm-ai-az-openai.openai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The release of ChatGPT played two key roles, according to the provided documents:

1. **Brought LLMs into the mainstream and boosted public awareness**  
   - ChatGPT is described as “the explosive charge that brought LLMs into the mainstream.”  
   - Its launch in 2022 “kicks off a huge increase in public awareness of LLMs and generative AI.”

2. **Greatly increased accessibility of advanced LLMs**  
   - It “turned GPT-3 and similar models into a service that is widely accessible to users through a web interface.”  
   - It “opened the door for anyone with internet access to interact with one of the most advanced LLMs through a simple web interface,” providing prompts and getting fast responses from high‑performing models (e.g., GPT‑3.5, GPT‑4).
